In [2]:
import numpy as np
import pandas as pd
from scipy.stats import norm

true_param = 5.0
size_n = 100
confidence_level = 0.95
alpha = 1 - confidence_level

np.random.seed(42)

X_data = np.random.uniform(true_param, true_param * 2, size=size_n)

print(f"Истинный параметр (theta): {true_param}")
print(f"Объем сгенерированной выборки: {size_n}")
print(f"Первые несколько значений: {np.round(X_data[:5], 3)}")

Истинный параметр (theta): 5.0
Объем сгенерированной выборки: 100
Первые несколько значений: [6.873 9.754 8.66  7.993 5.78 ]


In [3]:
x_max = np.max(X_data)

q_lower = (alpha / 2) ** (1 / size_n)
q_upper = (1 - alpha / 2) ** (1 / size_n)

exact_left = x_max / (1 + q_upper)
exact_right = x_max / (1 + q_lower)

print(f"Точный ДИ: ({exact_left:.4f}; {exact_right:.4f})")
print(f"Ширина интервала: {exact_right - exact_left:.4f}")

Точный ДИ: (4.9678; 5.0588)
Ширина интервала: 0.0910


In [4]:
x_bar = np.mean(X_data)
sample_std = np.std(X_data, ddof=1)
z_quant = norm.ppf(1 - alpha / 2)

theta_mm = (2 / 3) * x_bar

margin = z_quant * 2 * sample_std * np.sqrt(size_n - 1) / (3 * size_n)

asymp_left = theta_mm - margin
asymp_right = theta_mm + margin

print(f"Асимптотический ДИ: ({asymp_left:.4f}; {asymp_right:.4f})")
print(f"Ширина интервала: {asymp_right - asymp_left:.4f}")

Асимптотический ДИ: (4.7072; 5.0940)
Ширина интервала: 0.3868


In [6]:

num_iters = 1000 


mle_corr = x_max * (size_n + 1) / (2 * size_n + 1)


deltas_np = np.zeros(num_iters)

for i in range(num_iters):

    boot_x = np.random.choice(X_data, size=size_n, replace=True)
    boot_est = np.max(boot_x) * (size_n + 1) / (2 * size_n + 1)
    deltas_np[i] = boot_est - mle_corr


perc_left = np.percentile(deltas_np, alpha / 2 * 100)
perc_right = np.percentile(deltas_np, (1 - alpha / 2) * 100)

boot_np_left = mle_corr - perc_right
boot_np_right = mle_corr - perc_left

print(f"Непараметрический бутстрап ДИ: ({boot_np_left:.4f}; {boot_np_right:.4f})")
print(f"Ширина интервала: {boot_np_right - boot_np_left:.4f}")

Непараметрический бутстрап ДИ: (4.9919; 5.0453)
Ширина интервала: 0.0534


In [7]:

param_iters = 50_000
deltas_param = np.empty(param_iters) 

for j in range(param_iters):
    boot_x_param = np.random.uniform(mle_corr, 2 * mle_corr, size=size_n)
    boot_est_param = np.max(boot_x_param) * (size_n + 1) / (2 * size_n + 1)
    
    deltas_param[j] = boot_est_param - mle_corr

perc_param_left = np.percentile(deltas_param, alpha / 2 * 100)
perc_param_right = np.percentile(deltas_param, (1 - alpha / 2) * 100)

boot_p_left = mle_corr - perc_param_right
boot_p_right = mle_corr - perc_param_left

print(f"Параметрический бутстрап ДИ: ({boot_p_left:.4f}; {boot_p_right:.4f})")
print(f"Ширина интервала: {boot_p_right - boot_p_left:.4f}")

Параметрический бутстрап ДИ: (4.9677; 5.0590)
Ширина интервала: 0.0913


In [8]:
metrics =[
    {"Тип интервала": "Точный", "L_bound": exact_left, "R_bound": exact_right},
    {"Тип интервала": "Асимптотический (ОММ)", "L_bound": asymp_left, "R_bound": asymp_right},
    {"Тип интервала": "Непарам. бутстрап", "L_bound": boot_np_left, "R_bound": boot_np_right},
    {"Тип интервала": "Парам. бутстрап", "L_bound": boot_p_left, "R_bound": boot_p_right}
]

df_summary = pd.DataFrame(metrics)

df_summary["Длина"] = df_summary["R_bound"] - df_summary["L_bound"]

df_summary[f"Накрывает {true_param}"] = (df_summary["L_bound"] <= true_param) & (df_summary["R_bound"] >= true_param)

df_summary.round(4)

,Тип интервала,L_bound,R_bound,Длина,Накрывает 5.0
0,Точный,4.9678,5.0588,0.0910,True
1,Асимптотический (ОММ),4.7072,5.0940,0.3868,True
2,Непарам. бутстрап,4.9919,5.0453,0.0534,True
3,Парам. бутстрап,4.9677,5.0590,0.0913,True
